# 📊 Moon Data Pipeline (1900 - 2100)

This notebook explores different approaches to fetching lunar data over a 200-year span (1900 to 2100) using the MoonLora API. It documents how to safely scale API requests from simple yearly lookups up to heavy multi-decade daily extractions.

## 🔬 Part 1: Sample Testing (API Fetch & DataFrame Blueprint)

Before running massive loops, a single-point test ensures the API structure matches our target DataFrame format. Below is the blueprint of how a raw JSON response is parsed into a clean Pandas DataFrame row.

In [ ]:
#Importing Necesary Libraries
import pandas as pd
import requests
import time
from tqdm.notebook import tqdm

In [ ]:
# API Fetch Test
p = requests.get('https://moonlora.com/api/v1/phase?date=1969-07-20')
f = requests.get('https://moonlora.com/api/v1/full-moons/2026')
n = requests.get('https://moonlora.com/api/v1/new-moons/2026')

In [ ]:
# Creating DataFrame From JSON (phases)
phases = pd.DataFrame([p.json()])[['date','phase','illumination','moonAgeDays','moonSign','nextFullMoon','nextNewMoon']]

In [ ]:
phases.head()

,date,phase,illumination,moonAgeDays,moonSign,nextFullMoon,nextNewMoon
0,1969-07-20T12:00:00.000Z,first-quarter,0.332,5.77,Libra,1969-07-29T02:46:38.604Z,1969-08-13T05:17:55.306Z


In [ ]:
# Creating DataFrame From JSON (fullmoons)
fullmoons = pd.DataFrame(f.json()['fullMoons'])[['date','name','moonSign']]

In [ ]:
fullmoons.head()

,date,name,moonSign
0,2026-01-03T10:04:41.600Z,Wolf Moon,Cancer
1,2026-02-01T22:11:12.469Z,Snow Moon,Leo
2,2026-03-03T11:39:56.466Z,Worm Moon,Virgo
3,2026-04-02T02:14:07.754Z,Pink Moon,Libra
4,2026-05-01T17:25:29.598Z,Flower Moon,Scorpio


In [ ]:
# Creating DataFrame From JSON (newmoons)
newmoons = pd.DataFrame(n.json()['newMoons'])[['date', 'moonSign']]

In [ ]:
newmoons.head()

,date,moonSign
0,2026-01-18T19:53:42.734Z,Capricorn
1,2026-02-17T12:03:07.606Z,Aquarius
2,2026-03-19T01:25:52.757Z,Pisces
3,2026-04-17T11:54:13.780Z,Aries
4,2026-05-16T20:03:09.384Z,Taurus


## 📅 Part 2: Daily Moon Phases (The 73K Challenge)

This section tackles harvesting full daily data points for all 73,414 days in the target date range.

### 🛑 Code 1A: Original Version (Educational Reference Only)
* **Status:** ⚠️ Contains critical logical bugs and performance bottlenecks.
* **Why it fails/struggles:**
  1. **The Overwrite Bug:** The DataFrame conversion was placed *outside* the loop, meaning the temporary variable was overwritten 73,000+ times. Running this only saves the very last day (December 31, 2100).
  2. **Pandas Memory Overhead:** If the `.append()` statement is moved *inside* the loop, Pandas forces the system to copy the entire growing DataFrame into memory on every single iteration. This grinds execution speed to a halt and risks an Out-Of-Memory crash.
  3. **No Error Handling:** A single network hiccup or a rate-limit block (`429 Too Many Requests`) from the API server crashes the script instantly, destroying all unsaved data.

In [ ]:
# Creating Empty DataFrame (phase)
phase = pd.DataFrame()

In [ ]:
phase

""


In [ ]:
# WARNING: Functional but highly inefficient approach.
# While placing the DataFrame creation and append logic inside the innermost loop
# correctly captures all 73,414 days, executing `phase.append()` thousands of times
# creates massive memory fragmentation. Execution speed drops exponentially as the table grows.

for i in range(1900,2101):
  for j in range(1,13):
    if j in [1,3,5,7,8,10,12]:
        for k in range(1,32):
          p = requests.get('https://moonlora.com/api/v1/phase?date={}-{}-{}'.format(i,j,k))
          df = pd.DataFrame([p.json()])[['date','phase','illumination','moonAgeDays','moonSign','nextFullMoon','nextNewMoon']]
          phase = phase.append(df,ignore_index=True)
    elif j in [4,6,9,11]:
        for k in range(1,31):
          p = requests.get('https://moonlora.com/api/v1/phase?date={}-{}-{}'.format(i,j,k))
          df = pd.DataFrame([p.json()])[['date','phase','illumination','moonAgeDays','moonSign','nextFullMoon','nextNewMoon']]
          phase = phase.append(df,ignore_index=True)
    elif j == 2:
        if (i % 4 == 0 and i % 100 != 0) or (i % 400 == 0):
          for k in range(1,30):
            p = requests.get('https://moonlora.com/api/v1/phase?date={}-{}-{}'.format(i,j,k))
            df = pd.DataFrame([p.json()])[['date','phase','illumination','moonAgeDays','moonSign','nextFullMoon','nextNewMoon']]
            phase = phase.append(df,ignore_index=True)
        else:
          for k in range(1,29):
            p = requests.get('https://moonlora.com/api/v1/phase?date={}-{}-{}'.format(i,j,k))
            df = pd.DataFrame([p.json()])[['date','phase','illumination','moonAgeDays','moonSign','nextFullMoon','nextNewMoon']]
            phase = phase.append(df,ignore_index=True)



### ⚡ Code 1B: Optimized Daily Version (Recommended)
* **Status:** ✅ Fully functional, fast, and safe.
* **Key Improvements:**
  * **Native Date Handling:** Swaps complex nested loops for Python's native `datetime` library, automatically accounting for month lengths and leap years.
  * **Efficient Data Collection:** Appends lightweight dictionaries to a standard Python list inside the loop, converting to a Pandas DataFrame **only once** at the very end to save memory.
  * **Fault Tolerance & Backups:** Implements a `try-except` block to gracefully handle network issues, respects server rate limits (`429`), and auto-saves a backup CSV every 10,000 requests.
  * **Live Tracking:** Integrates `tqdm` to display a live progress bar with exact iterations per second (it/s) and estimated time remaining.

In [ ]:
import pandas as pd
import requests
from datetime import date, timedelta
from tqdm.notebook import tqdm
import time

# 1. Generate the exact list of dates natively
start_date = date(1900, 1, 1)
end_date = date(2100, 12, 31)
total_days = (end_date - start_date).days + 1

all_data = []

# 2. Loop with a progress bar so you can track the speed
for i in tqdm(range(total_days), desc="Fetching Moon Phases"):
    current_date = start_date + timedelta(days=i)
    date_str = current_date.strftime("%Y-%m-%d")
    url = f"https://moonlora.com/api/v1/phase?date={date_str}"

    try:
        response = requests.get(url, timeout=5)

        if response.status_code == 200:
            data = response.json()
            # Extract only the keys you need safely
            filtered_data = {k: data.get(k) for k in ['date', 'phase', 'illumination', 'moonAgeDays', 'moonSign', 'nextFullMoon', 'nextNewMoon']}
            all_data.append(filtered_data)
        elif response.status_code == 429:
            print(f"\nRate limited at {date_str}. Cooling down...")
            time.sleep(5)  # Wait 5 seconds if the server tells you to slow down

    except Exception as e:
        print(f"\nError fetching {date_str}: {e}")

    # Optional: Save a backup csv every 10,000 rows so you don't lose progress if Colab disconnects
    if len(all_data) % 10000 == 0 and len(all_data) > 0:
        pd.DataFrame(all_data).to_csv('moon_phases_backup.csv', index=False)

# 3. Convert the complete list to a final DataFrame
phase_df = pd.DataFrame(all_data)
phase_df.to_csv('moon_phases_1900_2100.csv', index=False)


Fetching Moon Phases:   0%|          | 0/73414 [00:00<?, ?it/s]

In [ ]:
phase_df.head()

,date,phase,illumination,moonAgeDays,moonSign,nextFullMoon,nextNewMoon
0,1900-01-01T12:00:00.000Z,new-moon,0.000,0.08,Capricorn,1900-01-15T19:07:18.344Z,1900-01-01T13:51:41.899Z
1,1900-01-02T12:00:00.000Z,new-moon,0.013,1.08,Capricorn,1900-01-15T19:07:18.344Z,1900-01-31T01:22:21.721Z
2,1900-01-03T12:00:00.000Z,waxing-crescent,0.048,2.08,Aquarius,1900-01-15T19:07:18.344Z,1900-01-31T01:22:21.721Z
3,1900-01-04T12:00:00.000Z,waxing-crescent,0.103,3.08,Aquarius,1900-01-15T19:07:18.344Z,1900-01-31T01:22:21.721Z
4,1900-01-05T12:00:00.000Z,waxing-crescent,0.177,4.08,Pisces,1900-01-15T19:07:18.344Z,1900-01-31T01:22:21.721Z


In [ ]:
# Checking The Shape of 'phase_df'
phase_df.shape

(73414, 7)

## 🌕 Part 3: Yearly Endpoints (Full Moons & New Moons)

Instead of manually sorting through daily loops to pinpoint specific lunar events, these blocks target the API's specialized yearly endpoints (`/full-moons/{year}` and `/new-moons/{year}`). This allows us to gather target events across the entire 200-year span in just **201 requests** per endpoint instead of over 73,000+.

### 1. Full Moons Data Fetcher
* **Endpoint:** `https://moonlora.com/api/v1/full-moons/{year}`
* **Fields Captured:** `date`, `name`, `moonSign`
* **Performance:** Highly efficient. Completed 201 iterations in ~6 seconds (~30 iterations/sec).

In [ ]:
# Creating Empty DataFrame (fullmoons)
fullmoons = pd.DataFrame()

In [ ]:
fullmoons

""


In [ ]:
# Loops through years 1900-2100 to pull annual arrays of full moon data.
# Note: pd.concat is used safely here because the loop scale is small (201 iterations).

for i in tqdm(range(1900,2101),desc="Fetching FullMoons: "):
  f = requests.get('https://moonlora.com/api/v1/full-moons/{}'.format(i))
  df = pd.DataFrame(f.json()['fullMoons'])[['date','name','moonSign']]
  fullmoons = pd.concat([fullmoons, df],ignore_index=True)

Fetching FullMoons:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
fullmoons.head(12)

,date,name,moonSign
0,1900-01-15T19:07:18.344Z,Wolf Moon,Cancer
1,1900-02-14T13:49:49.393Z,Snow Moon,Leo
2,1900-03-16T08:11:30.553Z,Worm Moon,Virgo
3,1900-04-15T01:01:51.312Z,Pink Moon,Libra
4,1900-05-14T15:36:18.471Z,Flower Moon,Scorpio
5,1900-06-13T03:38:16.407Z,Strawberry Moon,Sagittarius
6,1900-07-12T13:21:46.347Z,Buck Moon,Capricorn
7,1900-08-10T21:30:08.188Z,Sturgeon Moon,Aquarius
8,1900-09-09T05:06:27.445Z,Corn Moon,Pisces
9,1900-10-08T13:18:12.268Z,Hunter's Moon,Aries


In [ ]:
# Checking The Shape of 'phase_df'
fullmoons.shape

(2486, 3)

### 2. New Moons Data Fetcher
* **Endpoint:** `https://moonlora.com{year}`
* **Fields Captured:** `date`, `moonSign`
* **Performance:** Completed 201 iterations in ~10 seconds (~17 iterations/sec).

In [ ]:
newmoons = pd.DataFrame()

In [ ]:
newmoons

""


In [ ]:
# Loops through years 1900-2100 to pull annual arrays of new moon data.
# Note: pd.concat is used safely here because the loop scale is small (201 iterations).

for i in tqdm(range(1900,2101),desc="Fetching Newmoons: "):
  n = requests.get('https://moonlora.com/api/v1/new-moons/{}'.format(i))
  df = pd.DataFrame(n.json()['newMoons'])[['date', 'moonSign']]
  newmoons = pd.concat([newmoons, df], ignore_index=True)

Fetching Newmoons:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
newmoons.head(12)

,date,moonSign
0,1900-01-01T13:51:41.899Z,Capricorn
1,1900-01-31T01:22:21.721Z,Aquarius
2,1900-03-01T11:25:06.132Z,Pisces
3,1900-03-30T20:30:15.716Z,Aries
4,1900-04-29T05:23:02.969Z,Taurus
5,1900-05-28T14:49:34.852Z,Gemini
6,1900-06-27T01:27:24.741Z,Cancer
7,1900-07-26T13:43:04.471Z,Leo
8,1900-08-25T03:52:47.683Z,Virgo
9,1900-09-23T19:57:18.697Z,Libra


In [ ]:
# Checking The Shape of 'phase_df'
newmoons.shape

(2487, 2)

In [ ]:
# Converting fullmoons DataFrame & newmoons DataFrame to csv Files
fullmoons.to_csv('fullmoons.csv')
newmoons.to_csv('newmoons.csv')

### 💡 Optimization Note for Readers
While `pd.concat()` was used inside the yearly loops and worked perfectly fine there because the dataset is tiny (only 201 iterations), doing this on larger loops (like the daily tracker) will kill performance. For large-scale data harvesting, always stick to the **List-First, DataFrame-Last** approach!